**Legacy notebook.** Self-contained analysis code that predates the `src/mrvf` library and has not been ported to it. Kept for provenance and because it still produces figures in `results/`. Paths were updated to the `results/` layout; the next cell sets the working directory to the repository root, so run it from anywhere.

For the maintained pipeline see `notebooks/01_train_triple_regime.ipynb` and `notebooks/02_evaluate_rmse_vs_snr.ipynb`.

In [ ]:
import os
from pathlib import Path
# run from the repository root so ./results/... and ../subsamples resolve
_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "mrvf").is_dir())
os.chdir(_root)

# Notebook 1 (v4): DL Training & Simulation Evaluation — T2

## Changes in v4:
| # | Change |
|---|--------|
| 1 | Updated dictionary paths to `subsamples_v4` |
| 2 | DM and DL share the **same test set** (noise-free for noise-free eval, noisy per SNR for SNR eval) |
| 3 | DM always uses **noise-free dictionary** as template (no noisy-template variants) |
| 4 | Output saved to `./t2snr_results_v4/` to avoid any overlap with prior runs |


In [ ]:
import math
import os
import json
import time
import numpy as np
import scipy.io as sio
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# ── GPU Configuration ──
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'✓ GPU enabled: {torch.cuda.get_device_name(0)}')
    print(f'  CUDA version: {torch.version.cuda}')
    print(f'  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    device = torch.device('cpu')
    print('⚠ Running on CPU')
print(f'PyTorch: {torch.__version__}')


In [ ]:
import h5py

def load_mat(path, key):
    '''
    Load a matrix from a .mat file — supports both:
      - MATLAB v7.3 / HDF5  (scipy raises NotImplementedError → falls back to h5py)
      - MATLAB v5 / legacy  (scipy.io.loadmat)
    Returns a numpy array.
    '''
    try:
        mat = sio.loadmat(path)
        # Find the key (prefer requested, fall back to first non-dunder)
        if key in mat:
            return np.array(mat[key])
        cands = [k for k in mat if not k.startswith('_')]
        if cands:
            print(f'  ⚠ Key "{key}" not found in {path} — using "{cands[0]}"')
            return np.array(mat[cands[0]])
        raise KeyError(f'No valid key in {path}')
    except NotImplementedError:
        # MATLAB v7.3 HDF5 format
        with h5py.File(path, 'r') as f:
            if key in f:
                data = f[key][()]
            else:
                cands = [k for k in f.keys() if not k.startswith('#')]
                if not cands:
                    raise KeyError(f'No valid key in HDF5 file {path}')
                print(f'  ⚠ Key "{key}" not found in HDF5 {path} — using "{cands[0]}"')
                data = f[cands[0]][()]
            # HDF5 stores arrays transposed relative to MATLAB column-major
            if data.ndim >= 2:
                data = data.T
            return np.array(data, dtype=np.float32)


def load_param_mat(path, key='par'):
    '''Load parameter matrix and keep first 4 columns (SO2, CBV, R, T2).'''
    return load_mat(path, key)[:, :4]


print('✓ load_mat / load_param_mat defined (handles both v5 and v7.3 HDF5 .mat files)')


## 1. Configuration
**⚠️ Edit paths below.**


In [ ]:
# ═══════════════════════════════════════════════════
# CONFIGURATION — v4 (subsamples_v3)
# ═══════════════════════════════════════════════════
CONFIG = {
    # --- Paths (updated to subsamples_v3) ---
    "dict_base_path":     "../subsamples/subsamples_v3",
    "param_path":         "../subsamples/subsamples_v3/QuasiRand_par_t2_200.mat",
    "noisefree_sig_path": "../subsamples/subsamples_v3/QuasiRand_t2_200.mat",
    "output_dir":         "./results/t2snr_results_v4",

    # --- .mat keys ---
    "dict_key":  "Dico40_save",
    "param_key": "par_save",

    # --- SNR levels ---
    "snr_levels": [20, 50, 100, 150],

    # --- Parameter ranges: [SO2, CBV, R (m), T2 (s)] ---
    "param_mins": np.array([0.0,   0.0025,  1.0e-6,  0.050]),
    "param_maxs": np.array([1.0,   0.15,   25.0e-6,  0.200]),

    # --- Training ---
    "unified_n_samples":   1_600_000,
    "test_fraction":       0.15,
    "val_fraction":        0.15,
    "batch_size":          16384,
    "epochs":              150,
    "patience":            20,
    "lr":                  1e-5,
    "dropout":             0.1,
    "use_snr_token":       False,

    # --- Loss weights: [SO2, CBV, R, T2] ---
    "param_weights":       [1.5, 4.0, 3, 1],
}

os.makedirs(CONFIG['output_dir'], exist_ok=True)
os.makedirs(os.path.join(CONFIG['output_dir'], 'models'), exist_ok=True)
print('Output dir:', CONFIG['output_dir'])


In [ ]:
params = load_param_mat(CONFIG['param_path'], CONFIG['param_key'])
for i, name in enumerate(['SO2','CBV','R','T2']):
    print(f'{name}: min={params[:,i].min():.7f} max={params[:,i].max():.7f} '
          f'mean={params[:,i].mean():.7f} unique={len(np.unique(params[:,i]))}')

## 1b. Inspect .mat keys (run once to verify)


In [ ]:
def detect_key(mat, preferred):
    if preferred in mat:
        return preferred
    candidates = [k for k in mat if not k.startswith('_')]
    print(f'  ⚠ Key "{preferred}" not found — using "{candidates[0]}')
    return candidates[0]

par_mat = {'par': load_param_mat(CONFIG['param_path'], CONFIG['param_key'])}
nf_mat  = {CONFIG['dict_key']: load_mat(CONFIG['noisefree_sig_path'], CONFIG['dict_key'])}
print('=== Parameter file keys ===')
for k, v in par_mat.items():
    if not k.startswith('_'):
        print(f'  {k}: {v.shape}')
print('=== Noise-free signal file keys ===')
for k, v in nf_mat.items():
    if not k.startswith('_'):
        print(f'  {k}: {v.shape}')

# Check first SNR file
snr_path = os.path.join(CONFIG['dict_base_path'], 'QuasiRand_t2_snr20.mat')
if os.path.exists(snr_path):
    snr_mat = {CONFIG['dict_key']: load_mat(snr_path, CONFIG['dict_key'])}
    print('=== SNR=20 signal file keys ===')
    for k, v in snr_mat.items():
        if not k.startswith('_'):
            print(f'  {k}: {v.shape}')
else:
    print(f'⚠ {snr_path} not found')


## 2. Preprocessing Utilities


In [ ]:
def euclidean_norm_with_logamp(data):
    '''
    L2-normalise each row AND return the log of its norm as an extra feature.
    Preserves absolute-amplitude information discarded by plain L2-norm.
    Returns: (normalised_signals [N,40], log_amp [N,1])
    '''
    data = np.abs(data).astype(np.float32)
    norms = np.linalg.norm(data, axis=1, keepdims=True)
    norms = np.maximum(norms, 1e-12)
    log_amp = np.log(norms + 1e-6)
    return data / norms, log_amp

def euclidean_norm(data):
    data = np.abs(data).astype(np.float32)
    norms = np.linalg.norm(data, axis=1, keepdims=True)
    return data / np.maximum(norms, 1e-12)

def build_input(sig_norm, log_amp, snr=None, use_snr_token=True):
    '''
    Concatenate normalised signal + log-amplitude [+ SNR token].
    snr=None means noise-free (token = 0).
    Returns [N, 41] or [N, 42].
    '''
    log_amp_norm = (log_amp - 10.0) / 5.0
    x = np.concatenate([sig_norm, log_amp_norm], axis=1)
    if use_snr_token:
        if snr is None:
            token = np.zeros((len(x), 1), dtype=np.float32)
        else:
            token = np.full((len(x), 1), np.log10(float(snr)) / 3.0, dtype=np.float32)
        x = np.concatenate([x, token], axis=1)
    return x

def params_scale(params, mins, maxs):
    return ((params - mins) / (maxs - mins)).astype(np.float32)

def params_inverse(scaled, mins, maxs):
    return scaled * (maxs - mins) + mins

def filter_param_range(signals, params, mins, maxs):
    mask = np.ones(len(params), dtype=bool)
    for i in range(min(params.shape[1], 4)):
        mask &= (params[:, i] >= mins[i]) & (params[:, i] <= maxs[i])
    n_removed = np.sum(~mask)
    if n_removed:
        print(f'  Filtered {n_removed} out-of-range entries')
    return signals[mask], params[mask]

def clean_data(signals, params):
    valid = np.all(np.isfinite(signals), axis=1) & np.all(np.isfinite(params), axis=1)
    n_bad = np.sum(~valid)
    if n_bad:
        print(f'  Removed {n_bad} non-finite entries')
    return signals[valid], params[valid]

def compute_density_weights(y_scaled, n_bins=20, eps=1e-3):
    '''Inverse-density weights to up-weight underrepresented parameter combos.'''
    n = len(y_scaled)
    weights = np.ones(n, dtype=np.float32)
    for i in range(y_scaled.shape[1]):
        hist, edges = np.histogram(y_scaled[:, i], bins=n_bins, range=(0, 1))
        bin_idx = np.clip(np.digitize(y_scaled[:, i], edges[:-1]) - 1, 0, n_bins-1)
        density = hist[bin_idx].astype(np.float32) / n
        weights *= 1.0 / (density + eps)
    weights /= weights.mean()
    return weights

print('Utilities loaded.')


## 3. Data Loaders


In [ ]:
def prepare_gesfide_input(sig_norm):
    fid = sig_norm[:, :14]
    se  = sig_norm[:, 14:]
    fid_norm = fid / np.maximum(np.linalg.norm(np.abs(fid), axis=1, keepdims=True), 1e-12)
    se_norm  = se  / np.maximum(np.linalg.norm(np.abs(se),  axis=1, keepdims=True), 1e-12)
    log_ratio = np.log(
        np.maximum(np.mean(np.abs(fid), axis=1, keepdims=True), 1e-6) /
        np.maximum(np.mean(np.abs(se),  axis=1, keepdims=True), 1e-6)
    )
    return np.concatenate([fid_norm, se_norm, log_ratio], axis=1).astype(np.float32)


def prepare_unified_dataset(config):
    '''Mixed-SNR dataset: stack all SNR levels. Used for noisy model training.'''
    print('=' * 60)
    print('PREPARING UNIFIED MIXED-SNR DATASET')
    print('=' * 60)
    params_raw = load_param_mat(config['param_path'], config['param_key'])[:, :4]

    all_x, all_y, all_w = [], [], []
    for snr in config['snr_levels']:
        print(f'  Loading SNR={snr}...')
        sig_path = os.path.join(config['dict_base_path'], f'QuasiRand_t2_snr{snr}.mat')
        sig_raw  = load_mat(sig_path, config['dict_key'])
        sig_raw, params_i = filter_param_range(sig_raw, params_raw.copy(), config['param_mins'], config['param_maxs'])
        # sig_norm, log_amp = euclidean_norm_with_logamp(sig_raw)
        # sig_norm = euclidean_norm(sig_raw)
        # sig_norm, params_i = clean_data(sig_norm, params_i)
        sig_norm, params_i = clean_data(euclidean_norm(sig_raw), params_i)
        # x_i = prepare_gesfide_input(sig_norm)   # ← replaces euclidean_norm
        # log_amp = log_amp[:len(sig_norm)]
        # x_i = build_input(sig_norm, log_amp, snr=snr, use_snr_token=config['use_snr_token'])
        x_i = sig_norm
        y_i = params_scale(params_i, config['param_mins'], config['param_maxs'])
        w_i = compute_density_weights(y_i)
        w_i = np.ones(len(y_i), dtype=np.float32)
        all_x.append(x_i); all_y.append(y_i); all_w.append(w_i)
        print(f'    Loaded {len(x_i):,} samples')

    X = np.vstack(all_x); Y = np.vstack(all_y); W = np.concatenate(all_w)
    n_target = config['unified_n_samples']
    if len(X) > n_target:
        idx = np.random.choice(len(X), n_target, replace=False)
        X, Y, W = X[idx], Y[idx], W[idx]
    print(f'Total unified dataset: {len(X):,} samples, input dim={X.shape[1]}')

    x_tv, x_test, y_tv, y_test, w_tv, _ = train_test_split(
        X, Y, W, test_size=config['test_fraction'], random_state=42)
    x_train, x_val, y_train, y_val, w_train, _ = train_test_split(
        x_tv, y_tv, w_tv, test_size=config['val_fraction'], random_state=42)

    # Keep raw params for evaluation
    params_all = load_param_mat(config['param_path'], config['param_key'])[:, :4]
    y_test_raw = params_inverse(y_test, config['param_mins'][:4], config['param_maxs'][:4])

    print(f'  Train={len(x_train):,}  Val={len(x_val):,}  Test={len(x_test):,}')
    return x_train, x_val, x_test, y_train, y_val, y_test, y_test_raw, w_train


def prepare_noisefree_dataset(config):
    '''Noise-free dataset loader. Used for noise-free model training.'''
    print('=' * 60)
    print('PREPARING NOISE-FREE DATASET')
    print('=' * 60)
    sig_raw    = load_mat(config['noisefree_sig_path'], config['dict_key'])
    params_raw = load_param_mat(config['param_path'], config['param_key'])[:, :4]
    n = min(len(sig_raw), len(params_raw))
    # sig_raw, params_raw = sig_raw[:n], params_raw[:n]
    sig_raw, params_raw = filter_param_range(sig_raw, params_raw, config['param_mins'], config['param_maxs'])
    # sig_norm, log_amp = euclidean_norm_with_logamp(sig_raw)
    # sig_norm = euclidean_norm(sig_raw)
    # sig_norm, params_raw = clean_data(sig_norm, params_raw)
    sig_norm, params_raw = clean_data(euclidean_norm(sig_raw), params_raw)
    # X = prepare_gesfide_input(sig_norm)
    # log_amp = log_amp[:len(sig_norm)]
    # X = build_input(sig_norm, log_amp, snr=None, use_snr_token=config['use_snr_token'])
    X = sig_norm
    Y = params_scale(params_raw, config['param_mins'], config['param_maxs'])
    W = compute_density_weights(Y)
    W = np.ones(len(Y), dtype=np.float32)
    n_target = config['unified_n_samples']
    if len(X) > n_target:
        idx = np.random.choice(len(X), n_target, replace=False)
        X, Y, W = X[idx], Y[idx], W[idx]
    print(f'Noise-free dataset: {len(X):,} samples, input dim={X.shape[1]}')
    x_tv, x_test, y_tv, y_test, w_tv, _ = train_test_split(
        X, Y, W, test_size=config['test_fraction'], random_state=42)
    x_train, x_val, y_train, y_val, w_train, _ = train_test_split(
        x_tv, y_tv, w_tv, test_size=config['val_fraction'], random_state=42)
    y_test_raw = params_inverse(y_test, config['param_mins'][:4], config['param_maxs'][:4])
    print(f'  Train={len(x_train):,}  Val={len(x_val):,}  Test={len(x_test):,}')
    return x_train, x_val, x_test, y_train, y_val, y_test, y_test_raw, w_train


def prepare_snr_specific_dataset(config, snr):
    '''Load one SNR level — for cross-SNR testing. Uses same test split seed.'''
    print(f'--- SNR={snr} dataset ---')
    sig_path   = os.path.join(config['dict_base_path'], f'QuasiRand_t2_snr{snr}.mat')
    sig_raw    = load_mat(sig_path, config['dict_key'])
    params_raw = load_param_mat(config['param_path'], config['param_key'])[:, :4]
    n = min(len(sig_raw), len(params_raw))
    # sig_raw, params_raw = sig_raw[:n], params_raw[:n]
    sig_raw, params_raw = filter_param_range(sig_raw, params_raw, config['param_mins'], config['param_maxs'])
    # sig_norm, log_amp = euclidean_norm_with_logamp(sig_raw)
    # sig_norm = euclidean_norm(sig_raw)
    sig_norm, params_raw = clean_data(euclidean_norm(sig_raw), params_raw)
    # X = prepare_gesfide_input(sig_norm)
    # sig_norm, params_raw = clean_data(sig_norm, params_raw)
    # log_amp = log_amp[:len(sig_norm)]
    # X = build_input(sig_norm, log_amp, snr=snr, use_snr_token=config['use_snr_token'])
    X = sig_norm
    Y = params_scale(params_raw, config['param_mins'], config['param_maxs'])
    W = compute_density_weights(Y)
    W = np.ones(len(Y), dtype=np.float32)
    x_tv, x_test, y_tv, y_test, w_tv, _ = train_test_split(
        X, Y, W, test_size=config['test_fraction'], random_state=42)
    x_train, x_val, y_train, y_val, w_train, _ = train_test_split(
        x_tv, y_tv, w_tv, test_size=config['val_fraction'], random_state=42)
    y_test_raw = params_inverse(y_test, config['param_mins'][:4], config['param_maxs'][:4])
    return x_train, x_val, x_test, y_train, y_val, y_test, y_test_raw

print('Data loader functions defined.')


## 4. Model Architecture


In [ ]:
class Clamp01(nn.Module):
    '''Hard clamp to [0,1] — no saturation gradient unlike Sigmoid.'''
    def forward(self, x):
        return x.clamp(0.0, 1.0)


class Conv1DModel(nn.Module):
    '''
    Conv1D + MLP for MRvF parameter estimation.
    Input  : (batch, in_dim) — 40 echo signal + log-amp + optional SNR token
    Output : (batch, n_outputs) clamped to [0, 1]
    '''
    def __init__(self, n_outputs=4, dropout=0.3, in_dim=42):
        super().__init__()
        # self.conv_path = nn.Sequential(
        #     nn.Conv1d(1, 16, kernel_size=5, padding=2),
        #     nn.BatchNorm1d(16), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout),
        #     nn.Conv1d(16, 32, kernel_size=3, padding=1),
        #     nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout),
        #     nn.Conv1d(32, 64, kernel_size=3, padding=1),
        #     nn.BatchNorm1d(64), nn.ReLU(),
        # )
        # conv_out = 64 * 10  # 40 → pool → 20 → pool → 10
        # extra = in_dim - 40
        # self.mlp = nn.Sequential(
        #     nn.Linear(conv_out + extra, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout),
        #     nn.Linear(256, 128),              nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout),
        #     nn.Linear(128, n_outputs),
        # )
        self.conv_path = nn.Sequential(
            nn.Conv1d(1, 32,  kernel_size=7, padding=3),
            nn.BatchNorm1d(32),  nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout),
            nn.Conv1d(32, 64,  kernel_size=5, padding=2),
            nn.BatchNorm1d(64),  nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256), nn.ReLU(), nn.MaxPool1d(2),   # → 10
            nn.Conv1d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256), nn.ReLU(),
        )
        conv_out = 256 * 5   # 256 filters × 10 timesteps = 2560
        extra = in_dim - 40   # = 1 for 41-dim input
        
        # self.mlp = nn.Sequential(
        #     nn.Linear(conv_out + extra, 1024), nn.BatchNorm1d(1024), nn.ReLU(), nn.Dropout(dropout),
        #     nn.Linear(1024, 512),              nn.BatchNorm1d(512),  nn.ReLU(), nn.Dropout(dropout),
        #     nn.Linear(512,  256),              nn.BatchNorm1d(256),  nn.ReLU(), nn.Dropout(dropout),
        #     nn.Linear(256,  n_outputs),
        # )
        self.mlp = nn.Sequential(
            nn.Linear(conv_out + extra, 2048), nn.BatchNorm1d(2048), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(2048, 1024),             nn.BatchNorm1d(1024), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(1024, 512),              nn.BatchNorm1d(512),  nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512,  256),              nn.BatchNorm1d(256),  nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256,  n_outputs),
        )
        
        self.out_act = Clamp01()

    def forward(self, x):
        echo = x[:, :40].unsqueeze(1)  # [B, 1, 40]
        extra = x[:, 40:]              # [B, in_dim-40]
        c = self.conv_path(echo).flatten(1)  # [B, 640]
        h = torch.cat([c, extra], dim=1)
        return self.out_act(self.mlp(h))


# def make_model(config, n_outputs=4):
#     in_dim = 40 + 1 + (1 if config['use_snr_token'] else 0)
#     return Conv1DModel(n_outputs=n_outputs, dropout=config['dropout'], in_dim=in_dim).to(device)
def make_model(config, n_outputs=4):
    return Conv1DModel(n_outputs=n_outputs, dropout=config['dropout'], in_dim=40).to(device)
    # return Conv1DModel(n_outputs=n_outputs,
    #                    dropout=config['dropout'],
    #                    in_dim=41).to(device)   # ← 40 + 1 log-ratio


# Quick architecture check
_m = make_model(CONFIG)
_x = torch.randn(8, 40).to(device)
# _x = torch.randn(8, 41).to(device)
print(f'Model output shape: {_m(_x).shape}')
print(f'Parameters: {sum(p.numel() for p in _m.parameters()):,}')
del _m, _x


In [ ]:
class WeightedParamLoss(nn.Module):
    def __init__(self, param_weights):
        super().__init__()
        self.register_buffer('param_weights', torch.tensor(param_weights, dtype=torch.float32))

    def forward(self, pred, target, sample_weights=None):
        per_param  = torch.abs(pred - target) * self.param_weights.unsqueeze(0)
        per_sample = per_param.sum(dim=1)
        if sample_weights is not None:
            per_sample = per_sample * sample_weights
        return per_sample.mean()


def train_model(model, x_train, y_train, x_val, y_val, config,
                model_name='model', sample_weights=None):
    criterion = WeightedParamLoss(config['param_weights'][:y_train.shape[1]]).to(device)
    optimizer = optim.Adam(model.parameters(), lr=config['lr'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.7, min_lr=1e-7)
    

    x_t = torch.tensor(x_train, dtype=torch.float32).to(device)
    y_t = torch.tensor(y_train, dtype=torch.float32).to(device)
    x_v = torch.tensor(x_val,   dtype=torch.float32).to(device)
    y_v = torch.tensor(y_val,   dtype=torch.float32).to(device)
    # w_t = torch.tensor(sample_weights, dtype=torch.float32) if sample_weights is not None else None
    w_t = torch.tensor(sample_weights, dtype=torch.float32).to(device) if sample_weights is not None else None

    ds = TensorDataset(x_t, y_t) if w_t is None else TensorDataset(x_t, y_t, w_t)
    loader = DataLoader(ds, batch_size=config['batch_size'], shuffle=True, num_workers=0)

    # scheduler = optim.lr_scheduler.OneCycleLR(
    #     optimizer,
    #     max_lr=config['lr'],
    #     steps_per_epoch=len(loader),
    #     epochs=config['epochs'],
    #     pct_start=0.1,        # 10% warmup
    #     anneal_strategy='cos',
    # )

    best_val, patience_cnt, history = float('inf'), 0, {'loss': [], 'val_loss': []}
    model_path = os.path.join(config['output_dir'], 'models', f'{model_name}.pt')
    t0 = time.time()

    for epoch in range(config['epochs']):
        model.train()
        epoch_loss = 0.0
        for batch in loader:
            if len(batch) == 3:
                xb, yb, wb = [t.to(device) for t in batch]
            else:
                xb, yb = [t.to(device) for t in batch]; wb = None
            optimizer.zero_grad()
            loss = criterion(model(xb), yb, wb)
            loss.backward(); optimizer.step()
            # scheduler.step()
            epoch_loss += loss.item()
        train_loss = epoch_loss / len(loader)

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(x_v), y_v).item()
        scheduler.step(val_loss)
        history['loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        if val_loss < best_val:
            best_val = val_loss; patience_cnt = 0
            torch.save({'model_state_dict': model.state_dict(),
                        'n_outputs': y_train.shape[1],
                        'param_mins': config['param_mins'],
                        'param_maxs': config['param_maxs'],
                        'version': 'v4'},
                       model_path)
        else:
            patience_cnt += 1
        if patience_cnt >= config['patience']:
            print(f'  Early stop at epoch {epoch+1}')
            break
        # if (epoch + 1) % 5 == 0:
        #     print(f'  Epoch {epoch+1:3d}: train={train_loss:.5f}  val={val_loss:.5f}  best={best_val:.5f}')
        if (epoch + 1) % 5 == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(f'  Epoch {epoch+1:3d}: train={train_loss:.5f}  val={val_loss:.5f}  lr={current_lr:.2e}')

    elapsed = time.time() - t0
    print(f'  Training time: {elapsed:.1f}s  Best val loss: {best_val:.5f}')
    # Reload best checkpoint
    ckpt = torch.load(model_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    return history, elapsed

print('Training functions defined.')


In [ ]:
def evaluate_model(model, x_test, y_test, y_test_raw, config, n_outputs=4, label='model'):
    model.eval()
    param_names = ['SO2', 'CBV', 'R', 'T2'][:n_outputs]
    pscales = (config['param_maxs'] - config['param_mins'])[:n_outputs]

    preds = []
    with torch.no_grad():
        for start in range(0, len(x_test), 4096):
            xb = torch.tensor(x_test[start:start+4096], dtype=torch.float32, device=device)
            preds.append(model(xb).cpu().numpy())
    pred_scaled = np.vstack(preds)

    pred_phys = params_inverse(pred_scaled, config['param_mins'][:n_outputs], config['param_maxs'][:n_outputs])
    true_phys = y_test_raw[:, :n_outputs]

    results = {'y_pred_phys': pred_phys, 'y_true_phys': true_phys}
    print(f'\n── {label} ──')
    for i, pname in enumerate(param_names):
        rmse  = np.sqrt(np.mean((pred_phys[:, i] - true_phys[:, i])**2))
        nrmse = rmse / pscales[i]
        results[f'RMSE_{pname}'] = rmse
        results[f'NRMSE_{pname}'] = nrmse
        print(f'  {pname}: RMSE={rmse:.5f}  NRMSE={nrmse:.4f}')
    return results

print('Evaluation function defined.')


## 5. Load Datasets


In [ ]:
print('Loading noisy (mixed-SNR) dataset...')
(x_train_n, x_val_n, x_test_n,
 y_train_n, y_val_n, y_test_n,
 y_test_raw_n, w_train_n) = prepare_unified_dataset(CONFIG)

print('\nLoading noise-free dataset...')
(x_train_nf, x_val_nf, x_test_nf,
 y_train_nf, y_val_nf, y_test_nf,
 y_test_raw_nf, w_train_nf) = prepare_noisefree_dataset(CONFIG)

print(f'\nInput dim: {x_train_n.shape[1]}')


## 6. Train Models
### 6a. Noisy 4-param (mixed-SNR training)


In [ ]:
print(f'x_train_n shape: {x_train_n.shape}')  # should be (N, 41)
print(f'x_train_nf shape: {x_train_nf.shape}') # should be (N, 41)

In [ ]:
model_noisy_4p = make_model(CONFIG, n_outputs=4)
hist_noisy_4p, time_noisy_4p = train_model(
    model_noisy_4p, x_train_n, y_train_n, x_val_n, y_val_n, CONFIG,
    model_name='t2snr_noisy_4param_v4', sample_weights=w_train_n)


In [ ]:
res_noisy_4p = evaluate_model(model_noisy_4p, x_test_n,  y_test_n,  y_test_raw_n,  CONFIG, 4, 'Noisy_4param')

### 6b. Noisy 3-param (SO2, CBV, R only)


In [ ]:
model_noisy_3p = make_model(CONFIG, n_outputs=3)
hist_noisy_3p, time_noisy_3p = train_model(
    model_noisy_3p, x_train_n, y_train_n[:, :3], x_val_n, y_val_n[:, :3], CONFIG,
    model_name='t2snr_noisy_3param_v4', sample_weights=w_train_n)


### 6c. Noise-free 4-param


In [ ]:
model_nf_4p = make_model(CONFIG, n_outputs=4)
hist_nf_4p, time_nf_4p = train_model(
    model_nf_4p, x_train_nf, y_train_nf, x_val_nf, y_val_nf, CONFIG,
    model_name='t2snr_noisefree_4param_v4', sample_weights=w_train_nf)


### 6d. Noise-free 3-param


In [ ]:
model_nf_3p = make_model(CONFIG, n_outputs=3)
hist_nf_3p, time_nf_3p = train_model(
    model_nf_3p, x_train_nf, y_train_nf[:, :3], x_val_nf, y_val_nf[:, :3], CONFIG,
    model_name='t2snr_noisefree_3param_v4', sample_weights=w_train_nf)


## 7. Evaluate on Their Own Test Sets
Both DL models are evaluated on the **same** test split from their respective training sets.


In [ ]:
res_noisy_4p = evaluate_model(model_noisy_4p, x_test_n,  y_test_n,  y_test_raw_n,  CONFIG, 4, 'Noisy_4param')
res_noisy_3p = evaluate_model(model_noisy_3p, x_test_n,  y_test_n[:,:3], y_test_raw_n, CONFIG, 3, 'Noisy_3param')
res_nf_4p    = evaluate_model(model_nf_4p,    x_test_nf, y_test_nf, y_test_raw_nf, CONFIG, 4, 'NoiseFree_4param')
res_nf_3p    = evaluate_model(model_nf_3p,    x_test_nf, y_test_nf[:,:3], y_test_raw_nf, CONFIG, 3, 'NoiseFree_3param')


In [ ]:
res_noisy_4p = evaluate_model(model_noisy_4p, x_test_nf,  y_test_nf,  y_test_raw_nf,  CONFIG, 4, 'Noisy_4param')
res_noisy_3p = evaluate_model(model_noisy_3p, x_test_nf,  y_test_nf[:,:3], y_test_raw_nf, CONFIG, 3, 'Noisy_3param')
res_nf_4p    = evaluate_model(model_nf_4p,    x_test_nf, y_test_nf, y_test_raw_nf, CONFIG, 4, 'NoiseFree_4param')
res_nf_3p    = evaluate_model(model_nf_3p,    x_test_nf, y_test_nf[:,:3], y_test_raw_nf, CONFIG, 3, 'NoiseFree_3param')

In [ ]:
res_noisy_4p = evaluate_model(model_noisy_4p, x_test_nf,  y_test_nf,  y_test_raw_nf,  CONFIG, 4, 'Noisy_4param')
res_noisy_3p = evaluate_model(model_noisy_3p, x_test_nf,  y_test_nf[:,:3], y_test_raw_nf, CONFIG, 3, 'Noisy_3param')
res_nf_4p    = evaluate_model(model_nf_4p,    x_test_nf, y_test_nf, y_test_raw_nf, CONFIG, 4, 'NoiseFree_4param')
res_nf_3p    = evaluate_model(model_nf_3p,    x_test_nf, y_test_nf[:,:3], y_test_raw_nf, CONFIG, 3, 'NoiseFree_3param')

## 8. Dictionary Matching on Noise-Free Test Set
DM is evaluated on the **same noise-free test set** as the DL model.
The noise-free dictionary (no noisy templates) is used as the matching template.


In [ ]:
from sklearn.model_selection import train_test_split as _tts

def euclidean_norm(data):
    data = np.abs(data).astype(np.float32)
    norms = np.linalg.norm(data, axis=1, keepdims=True)
    return data / np.maximum(norms, 1e-12)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def gpu_dictionary_matching(dict_norm_np, test_norm_np, params, batch_size=2000):
    '''Inner-product DM on GPU. dict_norm and test_norm are already L2-normalised numpy arrays.'''
    dict_t = torch.tensor(dict_norm_np, dtype=torch.float16).to(device)  # (D, 40)
    best_idx = np.zeros(len(test_norm_np), dtype=np.int32)
    t0 = time.time()
    for start in range(0, len(test_norm_np), batch_size):
        end = min(start + batch_size, len(test_norm_np))
        batch = torch.tensor(test_norm_np[start:end], dtype=torch.float16).to(device)
        corr = torch.mm(batch, dict_t.T)          # (batch, D)
        best_idx[start:end] = corr.argmax(dim=1).cpu().numpy()
        del batch, corr
    print(f'    GPU DM done: {time.time()-t0:.1f}s')
    return params[best_idx]

print('Loading noise-free dictionary for DM...')
nf_sig_all = load_mat(CONFIG['noisefree_sig_path'], CONFIG['dict_key'])
nf_par_all = load_param_mat(CONFIG['param_path'], CONFIG['param_key'])[:, :4]
n_nf = min(len(nf_sig_all), len(nf_par_all))
nf_sig_all, nf_par_all = nf_sig_all[:n_nf], nf_par_all[:n_nf]

# Apply same filter as training
mask = np.ones(len(nf_par_all), dtype=bool)
for i in range(4):
    mask &= (nf_par_all[:, i] >= CONFIG['param_mins'][i]) & (nf_par_all[:, i] <= CONFIG['param_maxs'][i])
nf_sig_all, nf_par_all = nf_sig_all[mask], nf_par_all[mask]

valid = np.all(np.isfinite(nf_sig_all), axis=1)
nf_sig_all, nf_par_all = nf_sig_all[valid], nf_par_all[valid]
print(f'Dictionary: {nf_sig_all.shape[0]:,} entries')

# Reproduce exact same test split as prepare_noisefree_dataset
n_all = len(nf_sig_all)
idx_all = np.arange(n_all)
if n_all > CONFIG['unified_n_samples']:
    rng = np.random.default_rng(42)
    idx_all = rng.choice(n_all, CONFIG['unified_n_samples'], replace=False)
    nf_sig_sub = nf_sig_all[idx_all]
    nf_par_sub = nf_par_all[idx_all]
else:
    nf_sig_sub = nf_sig_all
    nf_par_sub = nf_par_all

_, nf_sig_test, _, nf_par_test = _tts(
    nf_sig_sub, nf_par_sub, test_size=CONFIG['test_fraction'], random_state=42)
print(f'Noise-free DM test set: {len(nf_sig_test):,} voxels')

# Run DM: match noise-free test against noise-free dictionary
dict_norm = euclidean_norm(nf_sig_all)
test_norm = euclidean_norm(nf_sig_test)

print('Running DM (noise-free test × noise-free dictionary)...')
t0 = time.time()
n_test = len(test_norm)
best_idx = np.zeros(n_test, dtype=np.int32)
chunk = 500
# for start in range(0, n_test, chunk):
#     end = min(start + chunk, n_test)
#     corr = np.dot(test_norm[start:end].astype(np.float16),
#                   dict_norm.T.astype(np.float16))
#     best_idx[start:end] = np.argmax(corr, axis=1)
# dm_pred_nf = nf_par_all[best_idx]
dm_pred_nf = gpu_dictionary_matching(dict_norm, test_norm, nf_par_all)

print(f'DM done in {time.time()-t0:.1f}s')

# RMSE
param_names_full = ['SO2', 'CBV', 'R', 'T2']
dm_nf_results = {}
print('\n── DM (noise-free) ──')
for i, pname in enumerate(param_names_full):
    rmse = float(np.sqrt(np.mean((dm_pred_nf[:, i] - nf_par_test[:, i])**2)))
    nrmse = rmse / (CONFIG['param_maxs'][i] - CONFIG['param_mins'][i])
    dm_nf_results[f'RMSE_{pname}'] = rmse
    dm_nf_results[f'NRMSE_{pname}'] = nrmse
    print(f'  {pname}: RMSE={rmse:.5f}  NRMSE={nrmse:.4f}')
dm_nf_results['y_pred_phys'] = dm_pred_nf
dm_nf_results['y_true_phys'] = nf_par_test


## 9. Cross-SNR Testing (DL vs DM on Same Noisy Test Set)
For each SNR: load noisy signals, split into the same test set, then:
- **DL** (noisy model): predict on the noisy test set
- **DL** (noise-free model): predict on the noisy test set (cross-SNR generalization)
- **DM**: match noisy test signals against **noise-free dictionary** (standard clinical scenario)


In [ ]:
snr_results = {}

for snr in CONFIG['snr_levels']:
    print(f'\n{"="*55}')
    print(f'SNR = {snr}')
    print(f'{"="*55}')

    # ── Load snr-specific dataset ──
    _, _, x_te_s, _, _, y_te_s, y_te_raw_s = prepare_snr_specific_dataset(CONFIG, snr)

    # ── DL noisy 4-param on this SNR ──
    r_ny4 = evaluate_model(model_noisy_4p, x_te_s, y_te_s, y_te_raw_s, CONFIG,
                           n_outputs=4, label=f'Noisy4p_SNR{snr}')
    r_ny4['snr'] = snr
    snr_results[f'noisy4p_snr{snr}'] = r_ny4

    # ── DL noise-free 4-param tested on noisy signals ──
    r_nf4 = evaluate_model(model_nf_4p, x_te_s, y_te_s, y_te_raw_s, CONFIG,
                           n_outputs=4, label=f'NF4p_on_SNR{snr}')
    r_nf4['snr'] = snr
    snr_results[f'nf4p_snr{snr}'] = r_nf4

    # ── DM: match noisy test against NOISE-FREE dictionary ──
    sig_path   = os.path.join(CONFIG['dict_base_path'], f'QuasiRand_t2_snr{snr}.mat')
    sig_raw    = load_mat(sig_path, CONFIG['dict_key'])
    params_snr = load_param_mat(CONFIG['param_path'], CONFIG['param_key'])[:, :4]
    n_s = min(len(sig_raw), len(params_snr))
    sig_raw, params_snr = sig_raw[:n_s], params_snr[:n_s]
    mask = np.ones(len(params_snr), dtype=bool)
    for i in range(4):
        mask &= (params_snr[:, i] >= CONFIG['param_mins'][i]) & (params_snr[:, i] <= CONFIG['param_maxs'][i])
    sig_raw, params_snr = sig_raw[mask], params_snr[mask]
    valid = np.all(np.isfinite(sig_raw), axis=1)
    sig_raw, params_snr = sig_raw[valid], params_snr[valid]

    _, snr_test_sig, _, snr_test_par = _tts(
        sig_raw, params_snr, test_size=CONFIG['test_fraction'], random_state=42)

    snr_test_norm = euclidean_norm(snr_test_sig)
    dm_best = np.zeros(len(snr_test_norm), dtype=np.int32)
    t0 = time.time()
    # for start in range(0, len(snr_test_norm), chunk):
    #     end = min(start + chunk, len(snr_test_norm))
    #     corr = np.dot(snr_test_norm[start:end].astype(np.float16),
    #                   dict_norm.T.astype(np.float16))
    #     dm_best[start:end] = np.argmax(corr, axis=1)
    # dm_pred_snr = nf_par_all[dm_best]
    dm_pred_snr = gpu_dictionary_matching(dict_norm, snr_test_norm, nf_par_all)
    print(f'  DM SNR={snr}: {time.time()-t0:.1f}s')

    r_dm = {'snr': snr, 'y_pred_phys': dm_pred_snr, 'y_true_phys': snr_test_par}
    print('  ── DM ──')
    for i, pname in enumerate(param_names_full):
        rmse = float(np.sqrt(np.mean((dm_pred_snr[:, i] - snr_test_par[:, i])**2)))
        nrmse = rmse / (CONFIG['param_maxs'][i] - CONFIG['param_mins'][i])
        r_dm[f'RMSE_{pname}'] = rmse
        r_dm[f'NRMSE_{pname}'] = nrmse
        print(f'    {pname}: RMSE={rmse:.5f}')
    snr_results[f'dm_snr{snr}'] = r_dm

print('\nCross-SNR testing complete.')


In [ ]:
snr_results = {}

for snr in CONFIG['snr_levels']:
    print(f'\n{"="*55}')
    print(f'SNR = {snr}')
    print(f'{"="*55}')

    # ── Load snr-specific dataset ──
    _, _, x_te_s, _, _, y_te_s, y_te_raw_s = prepare_snr_specific_dataset(CONFIG, snr)

    # ── DL noisy 4-param on this SNR ──
    r_ny4 = evaluate_model(model_noisy_4p, x_te_s, y_te_s, y_te_raw_s, CONFIG,
                           n_outputs=4, label=f'Noisy4p_SNR{snr}')
    r_ny4['snr'] = snr
    snr_results[f'noisy4p_snr{snr}'] = r_ny4

    # ── DL noise-free 4-param tested on noisy signals ──
    r_nf4 = evaluate_model(model_nf_4p, x_te_s, y_te_s, y_te_raw_s, CONFIG,
                           n_outputs=4, label=f'NF4p_on_SNR{snr}')
    r_nf4['snr'] = snr
    snr_results[f'nf4p_snr{snr}'] = r_nf4

    # ── DM: match noisy test against NOISE-FREE dictionary ──
    sig_path   = os.path.join(CONFIG['dict_base_path'], f'QuasiRand_t2_snr{snr}.mat')
    sig_raw    = load_mat(sig_path, CONFIG['dict_key'])
    params_snr = load_param_mat(CONFIG['param_path'], CONFIG['param_key'])[:, :4]
    n_s = min(len(sig_raw), len(params_snr))
    sig_raw, params_snr = sig_raw[:n_s], params_snr[:n_s]
    mask = np.ones(len(params_snr), dtype=bool)
    for i in range(4):
        mask &= (params_snr[:, i] >= CONFIG['param_mins'][i]) & (params_snr[:, i] <= CONFIG['param_maxs'][i])
    sig_raw, params_snr = sig_raw[mask], params_snr[mask]
    valid = np.all(np.isfinite(sig_raw), axis=1)
    sig_raw, params_snr = sig_raw[valid], params_snr[valid]

    _, snr_test_sig, _, snr_test_par = _tts(
        sig_raw, params_snr, test_size=CONFIG['test_fraction'], random_state=42)

    snr_test_norm = euclidean_norm(snr_test_sig)
    dm_best = np.zeros(len(snr_test_norm), dtype=np.int32)
    t0 = time.time()
    # for start in range(0, len(snr_test_norm), chunk):
    #     end = min(start + chunk, len(snr_test_norm))
    #     corr = np.dot(snr_test_norm[start:end].astype(np.float16),
    #                   dict_norm.T.astype(np.float16))
    #     dm_best[start:end] = np.argmax(corr, axis=1)
    # dm_pred_snr = nf_par_all[dm_best]
    dm_pred_snr = gpu_dictionary_matching(dict_norm, snr_test_norm, nf_par_all)
    print(f'  DM SNR={snr}: {time.time()-t0:.1f}s')

    r_dm = {'snr': snr, 'y_pred_phys': dm_pred_snr, 'y_true_phys': snr_test_par}
    print('  ── DM ──')
    for i, pname in enumerate(param_names_full):
        rmse = float(np.sqrt(np.mean((dm_pred_snr[:, i] - snr_test_par[:, i])**2)))
        nrmse = rmse / (CONFIG['param_maxs'][i] - CONFIG['param_mins'][i])
        r_dm[f'RMSE_{pname}'] = rmse
        r_dm[f'NRMSE_{pname}'] = nrmse
        print(f'    {pname}: RMSE={rmse:.5f}')
    snr_results[f'dm_snr{snr}'] = r_dm

print('\nCross-SNR testing complete.')


In [ ]:
# Run this after training to see if the model actually converged
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, (hist, title) in zip(axes, [
    (hist_noisy_4p, 'Noisy 4-param'),
    (hist_nf_4p,    'NoiseFree 4-param'),
]):
    ax.plot(hist['loss'],     label='Train')
    ax.plot(hist['val_loss'], label='Val')
    ax.set_title(title); ax.legend(); ax.set_xlabel('Epoch')
plt.tight_layout(); plt.show()

In [ ]:
params = load_param_mat(CONFIG['param_path'], 'par_save')
# Check R distribution at low CBV (CBV < 2%)
low_cbv = params[:, 1] < 0.02
print(f'Low CBV samples: {low_cbv.sum():,}')
print(f'R range at low CBV: {params[low_cbv, 2].min()*1e6:.1f} – {params[low_cbv, 2].max()*1e6:.1f} µm')
print(f'R unique at low CBV: {len(np.unique(params[low_cbv, 2]))}')

## 10. Save All Results


In [ ]:
all_dl_results = {
    't2snr_noisy_4param_v4':     res_noisy_4p,
    't2snr_noisy_3param_v4':     res_noisy_3p,
    't2snr_noisefree_4param_v4': res_nf_4p,
    't2snr_noisefree_3param_v4': res_nf_3p,
}
all_dl_results.update(snr_results)
all_dl_results['dm_noisefree_4param'] = dm_nf_results

metrics_dict = {}
for key, res in all_dl_results.items():
    metrics_dict[key] = {k: v for k, v in res.items()
                         if k not in ('y_pred_phys', 'y_true_phys', 'history')}

metrics_path = os.path.join(CONFIG['output_dir'], 'simulation_metrics_v5_0330.json')
with open(metrics_path, 'w') as f:
    json.dump(metrics_dict, f, indent=2, default=str)
print(f'Saved: {metrics_path}')

# Save predictions for figure notebook
pred_dict = {}
for key, res in all_dl_results.items():
    if 'y_pred_phys' in res:
        pred_dict[f'{key}_pred'] = res['y_pred_phys']
        pred_dict[f'{key}_true'] = res['y_true_phys']
preds_path = os.path.join(CONFIG['output_dir'], 'all_predictions_v5_0330.npz')
np.savez(preds_path, **pred_dict)
print(f'Saved: {preds_path}')


## 11. RMSE Summary Table


In [ ]:
pkeys = ['SO2', 'CBV', 'R', 'T2']
print(f'\n{"="*70}')
print('NOISE-FREE TEST SET: DL vs DM')
print(f'{"Model":<35}' + ''.join(f'{p:>10}' for p in pkeys))
print('-' * 75)
for label, res in [
    ('DL Noisy-4p (own test)',  res_noisy_4p),
    ('DL NoiseFree-4p (own test)', res_nf_4p),
    ('DM NoiseFree (same NF test)', dm_nf_results),
]:
    vals = ''.join(f"{res.get(f'RMSE_{p}', float('nan')):>10.5f}" for p in pkeys)
    print(f'{label:<35}{vals}')

print(f'\n{"="*70}')
print('CROSS-SNR (DL vs DM on SAME NOISY TEST SET, DM uses noise-free dict)')
print(f'{"SNR":<6}{"Method":<22}' + ''.join(f'{p:>10}' for p in pkeys))
print('-' * 75)
for snr in CONFIG['snr_levels']:
    for tag, key in [('DL-Noisy4p', f'noisy4p_snr{snr}'),
                     ('DL-NF4p',    f'nf4p_snr{snr}'),
                     ('DM-NF-dict', f'dm_snr{snr}')]:
        r = snr_results.get(key, {})
        vals = ''.join(f"{r.get(f'RMSE_{p}', float('nan')):>10.5f}" for p in pkeys)
        print(f'{snr:<6}{tag:<22}{vals}')
    print()


In [ ]:
pkeys = ['SO2', 'CBV', 'R', 'T2']
print(f'\n{"="*70}')
print('NOISE-FREE TEST SET: DL vs DM')
print(f'{"Model":<35}' + ''.join(f'{p:>10}' for p in pkeys))
print('-' * 75)
for label, res in [
    ('DL Noisy-4p (own test)',  res_noisy_4p),
    ('DL NoiseFree-4p (own test)', res_nf_4p),
    ('DM NoiseFree (same NF test)', dm_nf_results),
]:
    vals = ''.join(f"{res.get(f'RMSE_{p}', float('nan')):>10.5f}" for p in pkeys)
    print(f'{label:<35}{vals}')

print(f'\n{"="*70}')
print('CROSS-SNR (DL vs DM on SAME NOISY TEST SET, DM uses noise-free dict)')
print(f'{"SNR":<6}{"Method":<22}' + ''.join(f'{p:>10}' for p in pkeys))
print('-' * 75)
for snr in CONFIG['snr_levels']:
    for tag, key in [('DL-Noisy4p', f'noisy4p_snr{snr}'),
                     ('DL-NF4p',    f'nf4p_snr{snr}'),
                     ('DM-NF-dict', f'dm_snr{snr}')]:
        r = snr_results.get(key, {})
        vals = ''.join(f"{r.get(f'RMSE_{p}', float('nan')):>10.5f}" for p in pkeys)
        print(f'{snr:<6}{tag:<22}{vals}')
    print()


## 12. Diagnostic Plots


In [ ]:
import matplotlib.pyplot as plt

# Training curves
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('Training Curves (v3, subsamples_v3)', fontsize=13, fontweight='bold')
pairs = [
    (axes[0,0], hist_noisy_4p, 'Noisy 4-param'),
    (axes[0,1], hist_noisy_3p, 'Noisy 3-param'),
    (axes[1,0], hist_nf_4p,   'Noise-free 4-param'),
    (axes[1,1], hist_nf_3p,   'Noise-free 3-param'),
]
for ax, hist, title in pairs:
    ax.plot(hist['loss'],     label='Train', color='steelblue')
    ax.plot(hist['val_loss'], label='Val',   color='coral', linestyle='--')
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'training_curves_v4.png'), dpi=150, bbox_inches='tight')
plt.show()

# RMSE vs SNR
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('RMSE vs SNR: DL vs DM (DM uses noise-free template)', fontsize=12, fontweight='bold')
snrs = CONFIG['snr_levels']
for ax, pname in zip(axes, pkeys):
    ny4 = [snr_results.get(f'noisy4p_snr{s}', {}).get(f'RMSE_{pname}', float('nan')) for s in snrs]
    nf4 = [snr_results.get(f'nf4p_snr{s}', {}).get(f'RMSE_{pname}',   float('nan')) for s in snrs]
    dm  = [snr_results.get(f'dm_snr{s}', {}).get(f'RMSE_{pname}',     float('nan')) for s in snrs]
    ax.plot(snrs, ny4, 'o-',  color='steelblue', label='DL-Noisy4p')
    ax.plot(snrs, nf4, 's--', color='coral',     label='DL-NF4p')
    ax.plot(snrs, dm,  'd:',  color='seagreen',  label='DM (NF-dict)')
    ax.set_title(pname); ax.set_xlabel('SNR'); ax.set_ylabel('RMSE')
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'rmse_vs_snr_v4.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Training curves
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('Training Curves (v3, subsamples_v3)', fontsize=13, fontweight='bold')
pairs = [
    (axes[0,0], hist_noisy_4p, 'Noisy 4-param'),
    (axes[0,1], hist_noisy_3p, 'Noisy 3-param'),
    (axes[1,0], hist_nf_4p,   'Noise-free 4-param'),
    (axes[1,1], hist_nf_3p,   'Noise-free 3-param'),
]
for ax, hist, title in pairs:
    ax.plot(hist['loss'],     label='Train', color='steelblue')
    ax.plot(hist['val_loss'], label='Val',   color='coral', linestyle='--')
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'training_curves_v4.png'), dpi=150, bbox_inches='tight')
plt.show()

# RMSE vs SNR
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('RMSE vs SNR: DL vs DM (DM uses noise-free template)', fontsize=12, fontweight='bold')
snrs = CONFIG['snr_levels']
for ax, pname in zip(axes, pkeys):
    ny4 = [snr_results.get(f'noisy4p_snr{s}', {}).get(f'RMSE_{pname}', float('nan')) for s in snrs]
    nf4 = [snr_results.get(f'nf4p_snr{s}', {}).get(f'RMSE_{pname}',   float('nan')) for s in snrs]
    dm  = [snr_results.get(f'dm_snr{s}', {}).get(f'RMSE_{pname}',     float('nan')) for s in snrs]
    ax.plot(snrs, ny4, 'o-',  color='steelblue', label='DL-Noisy4p')
    ax.plot(snrs, nf4, 's--', color='coral',     label='DL-NF4p')
    ax.plot(snrs, dm,  'd:',  color='seagreen',  label='DM (NF-dict)')
    ax.set_title(pname); ax.set_xlabel('SNR'); ax.set_ylabel('RMSE')
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'rmse_vs_snr_v4.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
import h5py

def load_images_mat(path):
    '''Load images.mat — handles both MATLAB v5 and v7.3/HDF5.'''
    try:
        return sio.loadmat(path)
    except NotImplementedError:
        out = {}
        with h5py.File(path, 'r') as f:
            for k in f.keys():
                if k.startswith('#'): continue
                arr = np.array(f[k][()])
                if arr.ndim >= 2: arr = arr.T
                out[k] = arr
        return out

def euclidean_norm(data):
    data = np.abs(data).astype(np.float32)
    norms = np.linalg.norm(data, axis=1, keepdims=True)
    return data / np.maximum(norms, 1e-12)

def params_inverse(scaled, mins, maxs):
    return scaled * (maxs - mins) + mins

In [ ]:
import scipy.io as sio, nibabel as nib, re

# ── Load one subject's GESFIDE scan ──────────────────────────────────
images_mat  = '../code/images.mat'     # adjust path if needed
masks_dir   = '../GESFIDE_data/GES_ROI'
SUBJECT_KEY = 'img_e11air'             # ← change to any subject key

mat = load_images_mat(images_mat)
sig4d = np.asarray(mat[SUBJECT_KEY], dtype=np.float32)
if sig4d.ndim == 4 and sig4d.shape[0] == 40:
    sig4d = np.transpose(sig4d, (1, 2, 3, 0))
H, W, S, T = sig4d.shape
print(f'Signal shape: {H}×{W}×{S}×{T}')

# ── Load mask ─────────────────────────────────────────────────────────
subj_id = re.match(r'img_(e\d+)', SUBJECT_KEY, re.IGNORECASE).group(1).lower()
mask_path = next(
    p for sfx in ['_AIR','_air',''] for ext in ['.nii.gz','.nii']
    for p in [f'{masks_dir}/{subj_id}{sfx}_ROI{ext}']
    if __import__('os').path.exists(p)
)
mask3d = nib.load(mask_path).get_fdata()
if mask3d.ndim == 4: mask3d = mask3d[..., 0]

# ── Normalize signals ─────────────────────────────────────────────────
X      = sig4d.reshape(-1, 40).astype(np.float32)
X_norm = euclidean_norm(X)
mask_flat = mask3d.flatten().astype(bool)
finite_ok = np.isfinite(X_norm).all(axis=1)
proc_idx  = np.where(finite_ok & mask_flat)[0]
print(f'Valid voxels: {len(proc_idx):,}')

In [ ]:
# ── Pre-upload dictionary to GPU once ────────────────────────────────
dict_t_gpu = torch.tensor(dict_norm, dtype=torch.float16).to(device)
print(f'Dictionary on GPU: {dict_t_gpu.shape}')

def run_dl(model, n_out, name):
    pred = np.full((H*W*S, n_out), np.nan, dtype=np.float32)
    model.eval()
    with torch.no_grad():
        for start in range(0, len(proc_idx), 8192):
            sl  = proc_idx[start:start+8192]
            xb  = torch.tensor(X_norm[sl], dtype=torch.float32, device=device)
            pred[sl] = model(xb).cpu().numpy()
            del xb
    torch.cuda.empty_cache()
    phys = params_inverse(pred, CONFIG['param_mins'][:n_out],
                          CONFIG['param_maxs'][:n_out])
    phys_map = phys.reshape(H, W, S, n_out)
    phys_map[~mask3d.astype(bool)] = np.nan
    print(f'✓ {name} done')
    return phys_map


def run_dm(name='DM', batch_size=500):
    '''GPU inner-product DM.'''
    pred = np.full((H*W*S, 4), np.nan, dtype=np.float32)
    t0 = time.time()
    for start in range(0, len(proc_idx), batch_size):
        sl    = proc_idx[start:start+batch_size]
        try:
            batch = torch.tensor(X_norm[sl],
                                 dtype=torch.float16).to(device)
            corr  = torch.mm(batch, dict_t_gpu.T)
            best  = corr.argmax(dim=1).cpu().numpy()
            pred[sl] = nf_par_all[best]
            del batch, corr
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            batch_size = max(50, batch_size // 2)
            print(f'  OOM — reducing batch_size to {batch_size}')
            # retry same slice with smaller batch
            batch = torch.tensor(X_norm[sl],
                                 dtype=torch.float16).to(device)
            corr  = torch.mm(batch, dict_t_gpu.T)
            best  = corr.argmax(dim=1).cpu().numpy()
            pred[sl] = nf_par_all[best]
            del batch, corr
    torch.cuda.empty_cache()
    phys_map = pred.reshape(H, W, S, 4)
    phys_map[~mask3d.astype(bool)] = np.nan
    print(f'✓ {name} done in {time.time()-t0:.1f}s  ({len(proc_idx):,} voxels)')
    return phys_map


# ── Run all 5 methods ─────────────────────────────────────────────────
maps = {
    'Noisy-4p': run_dl(model_noisy_4p, 4, 'Noisy-4p'),
    'Noisy-3p': run_dl(model_noisy_3p, 3, 'Noisy-3p'),
    'NF-4p':    run_dl(model_nf_4p,    4, 'NF-4p'),
    'NF-3p':    run_dl(model_nf_3p,    3, 'NF-3p'),
    'DM':       run_dm(),
}

maps = {
    'Noisy-4p': run_dl(model_noisy_4p, 4, 'Noisy-4p'),
    'Noisy-3p': run_dl(model_noisy_3p, 3, 'Noisy-3p'),
    'NF-4p':    run_dl(model_nf_4p,    4, 'NF-4p'),
    'NF-3p':    run_dl(model_nf_3p,    3, 'NF-3p'),
    'DM':       run_dm(batch_size=200),
}

In [ ]:
# def run_dl(model, n_out, name):
#     pred = np.full((H*W*S, n_out), np.nan, dtype=np.float32)
#     model.eval()
#     with torch.no_grad():
#         for start in range(0, len(proc_idx), 8192):
#             sl = proc_idx[start:start+8192]
#             xb = torch.tensor(X_norm[sl], dtype=torch.float32, device=device)
#             pred[sl] = model(xb).cpu().numpy()
#     # Inverse scale to physical units
#     phys = params_inverse(pred, CONFIG['param_mins'][:n_out], CONFIG['param_maxs'][:n_out])
#     phys_map = phys.reshape(H, W, S, n_out)
#     # Zero outside mask
#     phys_map[~mask3d.astype(bool)] = np.nan
#     print(f'✓ {name} done')
#     return phys_map

# def run_dm(name='DM'):
#     pred = np.full((H*W*S, 4), np.nan, dtype=np.float32)
#     chunk = 500
#     dict_norm_np = dict_norm  # already computed in Step 8
#     for start in range(0, len(proc_idx), chunk):
#         sl = proc_idx[start:start+chunk]
#         corr = np.dot(X_norm[sl].astype(np.float16),
#                       dict_norm_np.T.astype(np.float16))
#         best = np.argmax(corr, axis=1)
#         pred[sl] = nf_par_all[best]
#     phys_map = pred.reshape(H, W, S, 4)
#     phys_map[~mask3d.astype(bool)] = np.nan
#     print(f'✓ {name} done')
#     return phys_map

# ── Pre-upload dictionary to GPU once ────────────────────────────────
dict_t_gpu = torch.tensor(dict_norm, dtype=torch.float16).to(device)
print(f'Dictionary on GPU: {dict_t_gpu.shape}')

def run_dl(model, n_out, name):
    pred = np.full((H*W*S, n_out), np.nan, dtype=np.float32)
    model.eval()
    with torch.no_grad():
        for start in range(0, len(proc_idx), 8192):
            sl  = proc_idx[start:start+8192]
            xb  = torch.tensor(X_norm[sl], dtype=torch.float32, device=device)
            pred[sl] = model(xb).cpu().numpy()
            del xb
    torch.cuda.empty_cache()
    phys = params_inverse(pred, CONFIG['param_mins'][:n_out],
                          CONFIG['param_maxs'][:n_out])
    phys_map = phys.reshape(H, W, S, n_out)
    phys_map[~mask3d.astype(bool)] = np.nan
    print(f'✓ {name} done')
    return phys_map


def run_dm(name='DM', batch_size=500):
    '''GPU inner-product DM.'''
    pred = np.full((H*W*S, 4), np.nan, dtype=np.float32)
    t0 = time.time()
    for start in range(0, len(proc_idx), batch_size):
        sl    = proc_idx[start:start+batch_size]
        try:
            batch = torch.tensor(X_norm[sl],
                                 dtype=torch.float16).to(device)
            corr  = torch.mm(batch, dict_t_gpu.T)
            best  = corr.argmax(dim=1).cpu().numpy()
            pred[sl] = nf_par_all[best]
            del batch, corr
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            batch_size = max(50, batch_size // 2)
            print(f'  OOM — reducing batch_size to {batch_size}')
            # retry same slice with smaller batch
            batch = torch.tensor(X_norm[sl],
                                 dtype=torch.float16).to(device)
            corr  = torch.mm(batch, dict_t_gpu.T)
            best  = corr.argmax(dim=1).cpu().numpy()
            pred[sl] = nf_par_all[best]
            del batch, corr
    torch.cuda.empty_cache()
    phys_map = pred.reshape(H, W, S, 4)
    phys_map[~mask3d.astype(bool)] = np.nan
    print(f'✓ {name} done in {time.time()-t0:.1f}s  ({len(proc_idx):,} voxels)')
    return phys_map


# ── Run all 5 methods ─────────────────────────────────────────────────
maps = {
    'Noisy-4p': run_dl(model_noisy_4p, 4, 'Noisy-4p'),
    'Noisy-3p': run_dl(model_noisy_3p, 3, 'Noisy-3p'),
    'NF-4p':    run_dl(model_nf_4p,    4, 'NF-4p'),
    'NF-3p':    run_dl(model_nf_3p,    3, 'NF-3p'),
    'DM':       run_dm(),
}

maps = {
    'Noisy-4p': run_dl(model_noisy_4p, 4, 'Noisy-4p'),
    'Noisy-3p': run_dl(model_noisy_3p, 3, 'Noisy-3p'),
    'NF-4p':    run_dl(model_nf_4p,    4, 'NF-4p'),
    'NF-3p':    run_dl(model_nf_3p,    3, 'NF-3p'),
    'DM':       run_dm(batch_size=200),
}

In [ ]:
SLICE = 4   # adjust slice index

param_vis = [
    ('SO₂ (%)',  0, 100,   (50, 90),  'hot'),
    ('CBV (%)',  1, 100,   (2,  10),  'turbo'),
    ('R (µm)',   2, 1e6,   (4,  20),  'pink'),
    ('T2 (ms)', 3, 1000,  (50, 130),  'viridis'),
]

method_names = list(maps.keys())   # 5 methods
n_rows = len(param_vis)
n_cols = len(method_names)

fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(3.2*n_cols, 3.2*n_rows),
                         gridspec_kw={'hspace':0.05, 'wspace':0.05})
fig.patch.set_facecolor('black')

for row, (pname, pidx, scale, (vmin,vmax), cmap) in enumerate(param_vis):
    for col, mname in enumerate(method_names):
        ax = axes[row, col]
        mmap = maps[mname]

        # Skip if model doesn't output this parameter (3-param models skip T2)
        if pidx >= mmap.shape[-1]:
            ax.set_facecolor('black'); ax.axis('off')
            ax.set_title(f'{mname}\n{pname}', fontsize=7, color='gray')
            continue

        img = mmap[:, :, SLICE, pidx] * scale
        cmap_obj = plt.cm.get_cmap(cmap).copy()
        cmap_obj.set_bad('black')
        ax.set_facecolor('black')

        im = ax.imshow(np.rot90(img), cmap=cmap_obj,
                       vmin=vmin, vmax=vmax, interpolation='nearest')
        ax.axis('off')

        if row == 0:
            ax.set_title(mname, fontsize=9, color='white', pad=4)
        if col == 0:
            ax.set_ylabel(pname, fontsize=8, color='white', labelpad=4)
            ax.yaxis.set_visible(True); ax.set_yticks([])

        if col == n_cols - 1:
            cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            cbar.ax.tick_params(labelsize=6, colors='white')
            cbar.outline.set_edgecolor('white')

fig.suptitle(f'Subject: {SUBJECT_KEY} | Slice {SLICE}',
             fontsize=11, color='white', fontweight='bold')
plt.savefig(os.path.join(CONFIG['output_dir'], f'invivo_comparison_{SUBJECT_KEY}_sl{SLICE}.png'),
            dpi=200, bbox_inches='tight', facecolor='black')
plt.show()

In [ ]:
SLICE = 4   # adjust slice index

param_vis = [
    ('SO₂ (%)',  0, 100,   (50, 90),  'hot'),
    ('CBV (%)',  1, 100,   (2,  10),  'turbo'),
    ('R (µm)',   2, 1e6,   (4,  20),  'pink'),
    ('T2 (ms)', 3, 1000,  (50, 130),  'viridis'),
]

method_names = list(maps.keys())   # 5 methods
n_rows = len(param_vis)
n_cols = len(method_names)

fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(3.2*n_cols, 3.2*n_rows),
                         gridspec_kw={'hspace':0.05, 'wspace':0.05})
fig.patch.set_facecolor('black')

for row, (pname, pidx, scale, (vmin,vmax), cmap) in enumerate(param_vis):
    for col, mname in enumerate(method_names):
        ax = axes[row, col]
        mmap = maps[mname]

        # Skip if model doesn't output this parameter (3-param models skip T2)
        if pidx >= mmap.shape[-1]:
            ax.set_facecolor('black'); ax.axis('off')
            ax.set_title(f'{mname}\n{pname}', fontsize=7, color='gray')
            continue

        img = mmap[:, :, SLICE, pidx] * scale
        cmap_obj = plt.cm.get_cmap(cmap).copy()
        cmap_obj.set_bad('black')
        ax.set_facecolor('black')

        im = ax.imshow(np.rot90(img), cmap=cmap_obj,
                       vmin=vmin, vmax=vmax, interpolation='nearest')
        ax.axis('off')

        if row == 0:
            ax.set_title(mname, fontsize=9, color='white', pad=4)
        if col == 0:
            ax.set_ylabel(pname, fontsize=8, color='white', labelpad=4)
            ax.yaxis.set_visible(True); ax.set_yticks([])

        if col == n_cols - 1:
            cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            cbar.ax.tick_params(labelsize=6, colors='white')
            cbar.outline.set_edgecolor('white')

fig.suptitle(f'Subject: {SUBJECT_KEY} | Slice {SLICE}',
             fontsize=11, color='white', fontweight='bold')
plt.savefig(os.path.join(CONFIG['output_dir'], f'invivo_comparison_{SUBJECT_KEY}_sl{SLICE}.png'),
            dpi=200, bbox_inches='tight', facecolor='black')
plt.show()

In [ ]:
# Load training params
params_raw = load_mat(CONFIG['param_path'], CONFIG['param_key'])
cbv_train  = params_raw[:, 1] * 100   # scale to %

print(f'Training CBV mean:   {cbv_train.mean():.2f}%')
print(f'Training CBV median: {np.median(cbv_train):.2f}%')

# Check where the flat band sits in predictions
cbv_pred = np.asarray(preds['t2snr_noisefree_4param_v4_pred'])[:, 1] * 100
cbv_true = np.asarray(preds['t2snr_noisefree_4param_v4_true'])[:, 1] * 100

# Find voxels in the flat band (predicted CBV in tight range)
band_mask = (cbv_pred > 4.0) & (cbv_pred < 5.5)
print(f'\nFlat band center:    {cbv_pred[band_mask].mean():.2f}%')
print(f'True CBV at band:    {cbv_true[band_mask].mean():.2f}%  '
      f'(range {cbv_true[band_mask].min():.1f}–{cbv_true[band_mask].max():.1f}%)')
print(f'\nDo they match? Training mean={cbv_train.mean():.2f}  '
      f'Band center={cbv_pred[band_mask].mean():.2f}')